In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers

In [ ]:
import torch
import math
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorWithPadding, set_seed
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm
import re
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
SEED = 42
BATCH_SIZE = 8
MAX_LENGTH = 512
STRIDE = 128
MLM_PROBABILITY = 0.15

In [ ]:
MODELS = [
    # "dbmdz/bert-base-italian-xxl-cased",
    # "linhd-postdata/alberti-bert-base-multilingual-cased",
    "mattiaferrarini/saba"
]
DATASET_PATH = "/content/drive/My Drive/italian_poems/italian_poems_test_rhymes.json"

In [ ]:
TARGET_LAYER = 11   # 0-indexed
TARGET_HEAD = 0     # 0-indexed

In [ ]:
def fix_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

fix_seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

In [ ]:
index = 14 #10 #19
sample = dataset[index]
text = sample["text"]
rhyme_tags = sample["rhyme_tags"]
print(f"Poem at index {index}:\n")
print(text)
print(f"\nRhyme Tags: {rhyme_tags}")

In [ ]:
filename = f"/content/drive/My Drive/saba/img/layer_{TARGET_LAYER}_head_{TARGET_HEAD}_poem_{index}.svg"
print(filename)

In [ ]:
def get_word_attention_grid(text, offsets, attention_weights):
    lines = text.split('\n')
    words_per_line = [line.split() for line in lines]
    max_words = max(len(w) for w in words_per_line) if words_per_line else 0

    attn_grid = np.full((len(lines), max_words), np.nan)
    word_grid = np.full((len(lines), max_words), "", dtype=object)

    current_char_pos = 0

    for row_idx, line in enumerate(lines):
        words_in_line = words_per_line[row_idx]
        start_col = max_words - len(words_in_line) # Right align

        line_start_pos = current_char_pos
        search_pos = 0

        for col_idx, word in enumerate(words_in_line):
            grid_col = start_col + col_idx

            match = re.search(re.escape(word), line[search_pos:])
            if not match: continue

            local_start, local_end = match.span()
            word_start = line_start_pos + search_pos + local_start
            word_end = line_start_pos + search_pos + local_end
            search_pos += local_end

            relevant_attn = []
            for i, (tok_start, tok_end) in enumerate(offsets):
                if tok_start == tok_end: continue
                if tok_end > word_start and tok_start < word_end:
                    relevant_attn.append(attention_weights[i])

            word_grid[row_idx, grid_col] = word
            if relevant_attn:
                attn_grid[row_idx, grid_col] = np.mean(relevant_attn)
            else:
                attn_grid[row_idx, grid_col] = 0.0

        current_char_pos += len(line) + 1

    return attn_grid, word_grid

In [ ]:
for model_id in MODELS:
    print(f"Processing: {model_id}...")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForMaskedLM.from_pretrained(model_id, output_attentions=True).to(device)
    model.eval()

    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", return_offsets_mapping=True)
    input_ids = inputs["input_ids"][0].clone()
    offsets = inputs["offset_mapping"][0]

    # Identify mask
    match = list(re.finditer(r"\b\w+\b", text))[-1]
    mask_start, mask_end = match.span()
    target_word = text[mask_start:mask_end]

    mask_indices = []
    for idx, (tok_start, tok_end) in enumerate(offsets):
        if tok_start == tok_end: continue
        if tok_start < mask_end and tok_end > mask_start:
            input_ids[idx] = tokenizer.mask_token_id
            mask_indices.append(idx)

    # Forward pass and head selection
    with torch.no_grad():
        outputs = model(input_ids.unsqueeze(0).to(device))
        layer_attentions = outputs.attentions[TARGET_LAYER]

        if TARGET_HEAD is not None:
            attentions = layer_attentions[:, TARGET_HEAD, :, :].squeeze(0).cpu()
            head_str = f"Head {TARGET_HEAD}"
        else:
            attentions = layer_attentions.mean(dim=1).squeeze(0).cpu()
            head_str = "Avg Heads"

    # Extract attention vector
    if mask_indices:
        attn_vector = attentions[mask_indices].mean(dim=0).numpy()
    else:
        print("Error: Mask not found.")
        continue

    # Build grid and plot
    attn_grid, word_grid = get_word_attention_grid(text, offsets, attn_vector)
    plt.figure(figsize=(12, 8))
    mask = np.isnan(attn_grid)

    ax = sns.heatmap(
        attn_grid,
        annot=word_grid,
        fmt="",
        cmap="Blues",
        mask=mask,
        cbar_kws={'label': 'Attention Weight'},
        linewidths=0.5,
        linecolor='lightgrey',
        square=False
    )

    plt.title(f"{model_id}\nLayer {TARGET_LAYER}, {head_str} -> Target: '{target_word}'", fontsize=14)
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()

    print(f"Displaying heatmap for {model_id} (Layer {TARGET_LAYER}, {head_str})...")
    plt.savefig(filename, format="svg", bbox_inches="tight")
    plt.show()

    del model
    torch.cuda.empty_cache()